In [16]:
import pandas as pd 
import pickle       
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import re
df = pd.read_csv('../data/car_web_scraped_dataset.csv')
df.head()


,name,year,miles,color,condition,price
0,Kia Forte,2022,"41,406 miles","Gray exterior, Black interior","No accidents reported, 1 Owner","$15,988"
1,Chevrolet Silverado 1500,2021,"15,138 miles","White exterior, Black interior","1 accident reported, 1 Owner","$38,008"
2,Toyota RAV4,2022,"32,879 miles","Silver exterior, Unknown interior","No accidents reported, 1 Owner","$24,988"
3,Honda Civic,2020,"37,190 miles","Blue exterior, Black interior","No accidents reported, 1 Owner","$18,998"
4,Honda Civic,2020,"27,496 miles","Black exterior, Black interior","No accidents reported, 1 Owner","$19,498"


In [17]:
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['miles'] = df['miles'].str.replace(' miles', '', regex=False).str.replace(',', '', regex=False).astype(float)

df[['exterior_color', 'interior_color']] = df['color'].str.split(', ', expand=True)
df['exterior_color'] = df['exterior_color'].str.replace(' exterior', '')
df['interior_color'] = df['interior_color'].str.replace(' interior', '')

def extract_accidents(text):
    if 'No accidents' in text: return 0
    match = re.search(r'(\d+)\s+accident', text)
    return int(match.group(1)) if match else 0

def extract_owners(text):
    match = re.search(r'(\d+)\s+Owner', text)
    return int(match.group(1)) if match else 1

df['accidents'] = df['condition'].apply(extract_accidents)
df['owners'] = df['condition'].apply(extract_owners)
df_clean = df.drop(columns=['color', 'condition'])
display(df_clean.head())

,name,year,miles,price,exterior_color,interior_color,accidents,owners
0,Kia Forte,2022,41406.0,15988.0,Gray,Black,0,1
1,Chevrolet Silverado 1500,2021,15138.0,38008.0,White,Black,1,1
2,Toyota RAV4,2022,32879.0,24988.0,Silver,Unknown,0,1
3,Honda Civic,2020,37190.0,18998.0,Blue,Black,0,1
4,Honda Civic,2020,27496.0,19498.0,Black,Black,0,1


In [ ]:
X = df_clean.drop(columns=['price'])
y = df_clean['price']

X_encoded = pd.get_dummies(X, drop_first=True)
pickle.dump(X_encoded.columns.tolist(), open('../model/carcolums_car.pkl', 'wb'))

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pickle.dump(scaler, open('../model/scaler_car.pkl', 'wb'))

In [ ]:
nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16), 
    activation='relu', 
    solver='adam', 
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

nn_model.fit(X_train_scaled, y_train)
y_pred = nn_model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"ความแม่นยำ: {r2 * 100:.2f}%")

pickle.dump(nn_model, open('../model/carscraped_model.pkl', 'wb'))

ความแม่นยำ: 85.42%


C:\Users\consl\AppData\Roaming\Python\Python314\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
